# 02 - MySQL Warehouse ETL

**Goal:** clean the source data, resolve customer identity, build dimensional and fact tables, save reproducible local extracts, and optionally load MySQL.

**Warehouse grain**

- `dim_customer`: one row per Olist `customer_unique_id`.
- `fact_orders`: one row per `(order_id, order_item_id)`.
- `fact_payments`: one row per `(order_id, payment_sequential)`.
- `fact_web_activity`: one clickstream event with an explicitly simulated customer link.
- `fact_customer_reviews`: Olist feedback connected to an Olist customer.
- `fact_product_reviews`: external product reviews with no customer-level Olist join.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid", palette="Set2")


def find_project_root() -> Path:
    """Find the project root whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter from the project root or notebooks directory.")


ROOT = find_project_root()
DATASETS = ROOT / "datasets"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

Project root: C:\Users\HP\Desktop\Customer 360 Intelligence


## 1. Load and type the Olist source tables

In [2]:
customers_raw = pd.read_csv(DATASETS / "olist_customers_dataset.csv")
orders_raw = pd.read_csv(DATASETS / "olist_orders_dataset.csv")
items_raw = pd.read_csv(DATASETS / "olist_order_items_dataset.csv")
payments_raw = pd.read_csv(DATASETS / "olist_order_payments_dataset.csv")
products_raw = pd.read_csv(DATASETS / "olist_products_dataset.csv")
translation = pd.read_csv(DATASETS / "product_category_name_translation.csv")
olist_reviews_raw = pd.read_csv(DATASETS / "olist_order_reviews_dataset.csv")

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for column in order_date_columns:
    orders_raw[column] = pd.to_datetime(orders_raw[column], errors="coerce")

assert orders_raw["order_id"].is_unique, "order_id must be unique in the order header."
assert not items_raw.duplicated(["order_id", "order_item_id"]).any(), "Order-item key is not unique."
assert not payments_raw.duplicated(["order_id", "payment_sequential"]).any(), "Payment key is not unique."

## 2. Build dimensions

When a repeat customer has multiple address records, the most recent order address is retained for the current customer profile. The count of source customer records remains available for lineage.

In [3]:
customer_history = (
    orders_raw[["customer_id", "order_purchase_timestamp"]]
    .merge(customers_raw, on="customer_id", how="left", validate="many_to_one")
    .sort_values("order_purchase_timestamp")
)
source_counts = (
    customers_raw.groupby("customer_unique_id")["customer_id"]
    .nunique()
    .rename("source_customer_count")
)
latest_customer = customer_history.drop_duplicates("customer_unique_id", keep="last")
dim_customer = (
    latest_customer.drop(columns="customer_id").rename(columns={
        "customer_unique_id": "customer_id",
        "customer_city": "city",
        "customer_state": "state",
        "customer_zip_code_prefix": "zip_code_prefix",
    })
    .merge(source_counts, left_on="customer_id", right_index=True, how="left")
    [["customer_id", "city", "state", "zip_code_prefix", "source_customer_count"]]
    .drop_duplicates("customer_id")
)

dim_product = (
    products_raw.merge(translation, on="product_category_name", how="left", validate="many_to_one")
    .rename(columns={
        "product_category_name": "category_name",
        "product_category_name_english": "category_name_english",
    })
    [[
        "product_id", "category_name", "category_name_english", "product_weight_g",
        "product_length_cm", "product_height_cm", "product_width_cm",
    ]]
    .drop_duplicates("product_id")
)
dim_product["category_name_english"] = dim_product["category_name_english"].fillna("unknown")

date_range = pd.date_range(
    orders_raw["order_purchase_timestamp"].min().normalize(),
    orders_raw["order_purchase_timestamp"].max().normalize(),
    freq="D",
)
dim_date = pd.DataFrame({"full_date": date_range})
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["calendar_year"] = dim_date["full_date"].dt.year
dim_date["calendar_quarter"] = dim_date["full_date"].dt.quarter
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["week_number"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["day_of_month"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["full_date"].dt.dayofweek.ge(5).astype(int)
dim_date = dim_date[[
    "date_key", "full_date", "calendar_year", "calendar_quarter", "month_number",
    "month_name", "week_number", "day_of_month", "day_name", "is_weekend",
]]

print("dim_customer", dim_customer.shape)
print("dim_product", dim_product.shape)
print("dim_date", dim_date.shape)

dim_customer (96096, 5)
dim_product (32951, 7)
dim_date (774, 10)


## 3. Build Olist transaction and feedback facts

`revenue` is merchandise value (`price`). Freight is stored separately, and `gross_value` is merchandise plus freight. Canceled and unavailable orders remain in the warehouse for auditability but are excluded from revenue analytics later.

In [4]:
order_header = (
    orders_raw.merge(
        customers_raw[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left",
        validate="one_to_one",
    )
    .rename(columns={
        "customer_id": "source_customer_id",
        "customer_unique_id": "customer_id",
        "order_purchase_timestamp": "purchase_date",
        "order_delivered_customer_date": "delivered_date",
        "order_estimated_delivery_date": "estimated_delivery_date",
    })
)

fact_orders = order_header.merge(items_raw, on="order_id", how="inner", validate="one_to_many")
fact_orders = fact_orders.rename(columns={"price": "item_price"})
fact_orders["purchase_date_key"] = fact_orders["purchase_date"].dt.strftime("%Y%m%d").astype(int)
fact_orders["quantity"] = 1
fact_orders["revenue"] = fact_orders["item_price"]
fact_orders["gross_value"] = fact_orders["item_price"] + fact_orders["freight_value"]
fact_orders = fact_orders[[
    "order_id", "order_item_id", "source_customer_id", "customer_id", "product_id",
    "seller_id", "order_status", "purchase_date", "purchase_date_key", "delivered_date",
    "estimated_delivery_date", "quantity", "item_price", "freight_value", "revenue", "gross_value",
]]

fact_payments = payments_raw[[
    "order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value",
]].copy()

fact_customer_reviews = (
    olist_reviews_raw.merge(
        order_header[["order_id", "customer_id"]],
        on="order_id",
        how="inner",
        validate="many_to_one",
    )
    .rename(columns={
        "review_creation_date": "review_date",
        "review_score": "rating",
        "review_comment_title": "review_title",
        "review_comment_message": "review_text",
    })
)
fact_customer_reviews["review_date"] = pd.to_datetime(fact_customer_reviews["review_date"], errors="coerce")
fact_customer_reviews = fact_customer_reviews[[
    "review_id", "order_id", "customer_id", "review_date", "rating", "review_title", "review_text",
]].drop_duplicates(["review_id", "order_id"])

print("fact_orders", fact_orders.shape)
print("fact_payments", fact_payments.shape)
print("fact_customer_reviews", fact_customer_reviews.shape)

fact_orders (112650, 16)
fact_payments (103886, 5)
fact_customer_reviews (99224, 7)


## 4. Create the simulated clickstream identity-resolution layer

The clickstream and Olist IDs come from different businesses. This deterministic one-to-one mapping is a **demonstration layer**, not a discovered real-world identity match. Products are also not joined across the two ecosystems.

Set `CLICKSTREAM_ROWS = None` only if the machine has enough memory to process the complete December file.

In [5]:
CLICKSTREAM_ROWS = 300_000
click_columns = [
    "event_time", "event_type", "product_id", "category_code", "brand",
    "price", "user_id", "user_session",
]
click = pd.read_csv(
    DATASETS / "2019-Dec.csv",
    usecols=click_columns,
    nrows=CLICKSTREAM_ROWS,
    low_memory=False,
)
click["event_time"] = pd.to_datetime(click["event_time"], errors="coerce", utc=True).dt.tz_localize(None)
click["click_user_id"] = click["user_id"].astype("string")

click_users = np.sort(click["click_user_id"].dropna().unique())
canonical_customers = np.sort(dim_customer["customer_id"].unique())
mapped_count = min(len(click_users), len(canonical_customers))
rng_mapping = np.random.default_rng(42)
mapped_customers = rng_mapping.choice(canonical_customers, size=mapped_count, replace=False)
customer_identity_mapping = pd.DataFrame({
    "click_user_id": click_users[:mapped_count],
    "customer_id": mapped_customers,
    "identity_link_is_simulated": 1,
})

fact_web_activity = (
    click.merge(customer_identity_mapping, on="click_user_id", how="inner", validate="many_to_one")
    .rename(columns={"product_id": "source_product_id"})
    [[
        "customer_id", "click_user_id", "event_time", "event_type", "source_product_id",
        "category_code", "brand", "price", "user_session", "identity_link_is_simulated",
    ]]
    .dropna(subset=["event_time", "event_type"])
)
customer_identity_mapping.to_csv(PROCESSED / "customer_identity_mapping.csv", index=False)
print("fact_web_activity", fact_web_activity.shape)

fact_web_activity (300000, 10)


## 5. Build the external product-review fact

Datafiniti's product `id` repeats across reviews and therefore cannot be the review primary key. A stable SHA-256 key is created from the source product, date, username, and review text. Sentiment labels here are rating-derived training labels; the NLP model is trained in Notebook 05.

In [6]:
import hashlib

amazon_columns = [
    "id", "name", "categories", "reviews.date", "reviews.rating",
    "reviews.text", "reviews.username",
]
amazon = pd.read_csv(DATASETS / "1429_1.csv", usecols=amazon_columns, low_memory=False)
amazon = amazon.rename(columns={
    "id": "source_product_id",
    "name": "product_name",
    "categories": "category_name",
    "reviews.date": "review_date",
    "reviews.rating": "rating",
    "reviews.text": "review_text",
    "reviews.username": "review_username",
})
amazon["review_date"] = pd.to_datetime(amazon["review_date"], errors="coerce", utc=True).dt.tz_localize(None)
amazon["category_name"] = (
    amazon["category_name"].fillna("unknown").str.split(",").str[0].str.lower().str.strip()
)
key_columns = ["source_product_id", "review_date", "review_username", "review_text"]
key_parts = amazon[key_columns].astype("object")
key_parts = key_parts.where(key_parts.notna(), "").astype(str)
key_text = (
    key_parts["source_product_id"] + "|" + key_parts["review_date"] + "|"
    + key_parts["review_username"] + "|" + key_parts["review_text"]
)
amazon["review_key"] = key_text.map(
    lambda value: hashlib.sha256(str(value).encode("utf-8")).hexdigest()
)
amazon["sentiment_score"] = ((amazon["rating"] - 3) / 2).clip(-1, 1).fillna(0)
amazon["sentiment_label"] = np.select(
    [amazon["rating"].le(2), amazon["rating"].eq(3), amazon["rating"].ge(4)],
    ["Negative", "Neutral", "Positive"],
    default="Unknown",
)
fact_product_reviews = amazon[[
    "review_key", "source_product_id", "product_name", "category_name", "review_date",
    "rating", "review_text", "sentiment_label", "sentiment_score",
]].drop_duplicates("review_key")
print("fact_product_reviews", fact_product_reviews.shape)

fact_product_reviews (34660, 9)


## 6. Generate transparent synthetic campaign events

The Olist dataset has no marketing table. These records exist only to demonstrate campaign funnel and ROI analytics, and `is_synthetic = 1` keeps that limitation visible.

In [7]:
rng = np.random.default_rng(42)
campaign_ids = [f"CAMP_{number:03d}" for number in range(1, 9)]
dim_campaign = pd.DataFrame({
    "campaign_id": campaign_ids,
    "campaign_type": ["Email", "SMS", "Coupon", "Loyalty", "Retargeting", "Referral", "Winback", "Seasonal"],
    "campaign_start_date": pd.date_range("2018-01-01", periods=8, freq="30D"),
    "campaign_cost": rng.integers(25_000, 120_000, size=8).astype(float),
})

campaign_event_count = 60_000
fact_campaign = pd.DataFrame({
    "campaign_id": rng.choice(campaign_ids, size=campaign_event_count),
    "customer_id": rng.choice(dim_customer["customer_id"], size=campaign_event_count, replace=True),
})
fact_campaign["email_sent"] = 1
fact_campaign["opened"] = rng.binomial(1, 0.55, campaign_event_count)
fact_campaign["clicked"] = fact_campaign["opened"] * rng.binomial(1, 0.35, campaign_event_count)
fact_campaign["converted"] = fact_campaign["clicked"] * rng.binomial(1, 0.22, campaign_event_count)
generated_revenue = rng.normal(180, 65, campaign_event_count).clip(25)
fact_campaign["revenue_generated"] = np.where(fact_campaign["converted"].eq(1), generated_revenue, 0).round(2)
fact_campaign["campaign_date"] = pd.to_datetime(
    rng.choice(pd.date_range("2018-01-01", "2018-08-31"), size=campaign_event_count)
).date
fact_campaign["is_synthetic"] = 1
display(fact_campaign.head())

,campaign_id,customer_id,email_sent,opened,clicked,converted,revenue_generated,campaign_date,is_synthetic
0,CAMP_002,ce60ceaf2d12ac87fa2119a364c89e81,1,0,0,0,0.00,2018-06-08,1
1,CAMP_001,7e05679b11824f00a2ab44b5fe6934c7,1,0,0,0,0.00,2018-06-04,1
2,CAMP_005,15b2459300f0b85b233d798c23552cee,1,1,1,1,154.58,2018-08-01,1
3,CAMP_008,5f28e0db75ddc9aec85f8c30533b95ba,1,0,0,0,0.00,2018-08-03,1
4,CAMP_006,8deaaff82a1e3dec44a5e59ed5f67574,1,1,0,0,0.00,2018-01-30,1


## 7. Validate and save the processed warehouse extracts

In [8]:
tables = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_date": dim_date,
    "fact_orders": fact_orders,
    "fact_payments": fact_payments,
    "fact_web_activity": fact_web_activity,
    "fact_customer_reviews": fact_customer_reviews,
    "fact_product_reviews": fact_product_reviews,
    "dim_campaign": dim_campaign,
    "fact_campaign": fact_campaign,
}

assert dim_customer["customer_id"].is_unique
assert not fact_orders.duplicated(["order_id", "order_item_id"]).any()
assert set(fact_orders["customer_id"]).issubset(set(dim_customer["customer_id"]))
assert set(fact_orders["product_id"]).issubset(set(dim_product["product_id"]))
assert fact_orders["revenue"].ge(0).all()

for name, frame in tables.items():
    frame.to_csv(PROCESSED / f"{name}.csv", index=False)
    print(f"{name:26s} {frame.shape}")

dim_customer               (96096, 5)
dim_product                (32951, 7)
dim_date                   (774, 10)


fact_orders                (112650, 16)


fact_payments              (103886, 5)


fact_web_activity          (300000, 10)


fact_customer_reviews      (99224, 7)


fact_product_reviews       (34660, 9)
dim_campaign               (8, 4)
fact_campaign              (60000, 9)


## 8. Optional MySQL load

1. Run `sql/schema.sql` in MySQL Workbench first.
2. Set `LOAD_TO_MYSQL = True` below.
3. Put credentials in environment variables, or enter the password securely when prompted.

The notebook clears only the project tables before loading, making reruns deterministic.

In [9]:
import os

LOAD_TO_MYSQL = os.getenv("LOAD_TO_MYSQL", "false").strip().lower() in {"1", "true", "yes"}

if LOAD_TO_MYSQL:
    import getpass
    from sqlalchemy import URL, create_engine, text

    mysql_url = URL.create(
        drivername="mysql+pymysql",
        username=os.getenv("MYSQL_USER", "root"),
        password=os.getenv("MYSQL_PASSWORD") or getpass.getpass("MySQL password: "),
        host=os.getenv("MYSQL_HOST", "localhost"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
        database=os.getenv("MYSQL_DATABASE", "customer360_db"),
    )
    engine = create_engine(mysql_url, pool_pre_ping=True)

    delete_order = [
        "fact_campaign", "fact_product_reviews", "fact_customer_reviews", "fact_web_activity",
        "fact_payments", "fact_orders", "dim_campaign", "dim_date", "dim_product", "dim_customer",
    ]
    with engine.begin() as connection:
        for table_name in delete_order:
            connection.execute(text(f"DELETE FROM {table_name}"))

    load_order = [
        "dim_customer", "dim_product", "dim_date", "dim_campaign", "fact_orders",
        "fact_payments", "fact_web_activity", "fact_customer_reviews",
        "fact_product_reviews", "fact_campaign",
    ]
    for table_name in load_order:
        tables[table_name].to_sql(
            table_name,
            engine,
            if_exists="append",
            index=False,
            chunksize=1_000,
        )
        print(f"Loaded {table_name}: {len(tables[table_name]):,} rows")
else:
    print("Local extracts are ready. MySQL load was skipped.")

Local extracts are ready. MySQL load was skipped.
